## **Generative AI: LLM-Based Dialogue Summarization with FLAN-T5 Model**

### Introduction

In this project, we explore the use of **large language models (LLMs)** for **dialogue summarization** using Python and the HuggingFace Transformers library. We utilize the **FLAN-T5-Base** model and the **DialogSum dataset**, which contains dialogues paired with human-written summaries. It is based on the lab exercise of Deeplearning.AI and AWS's course on Generative AI with LLMs.

The project demonstrates different summarization strategies:

- **Baseline generation** without instructions  
- **Zero-shot prompting** using instructional prompts  
- **One-shot prompting** with a single example  
- **Few-shot prompting** with multiple examples  

Through these experiments, we examine how **prompt design and example-based guidance** affect the model's ability to generate accurate and contextually relevant summaries. The goal is to create a concise, human-like summary of dialogues using modern LLM techniques.


### Importing Required Libraries

This cell imports all the essential libraries needed for building the dialogue-summarization project:

- **load_dataset** from `datasets`: used to load the DialogSum dataset from HuggingFace.
- **AutoModelForSeq2SeqLM** from `transformers`: loads a pretrained sequence-to-sequence language model (FLAN-T5 in this project).
- **AutoTokenizer** from `transformers`: loads the tokenizer that converts text into model-readable tokens and back to text.
- **GenerationConfig**: allows customization of generation behavior such as temperature, max tokens, or sampling settings.

These tools together enable loading data, preparing text, running the model, and controlling how summaries are generated.


In [1]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import GenerationConfig

### Loading the DialogSum Dataset

Here we specify the name of the dataset we want to use from HuggingFace:  
`"knkarthick/dialogsum"`, a dataset containing dialogues and their human-written summaries.

The `load_dataset()` function downloads and loads the dataset into the `dataset` variable, giving us access to its `train`, `validation`, and `test` splits for use in our summarization experiments.


In [2]:
huggingface_dataset_name = "knkarthick/dialogsum"

dataset = load_dataset(huggingface_dataset_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.csv:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [3]:
#check the dataset loaded
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

It contains train, validation and test datasets. Each contains id, dialogue, summary and topic.

In [4]:
#extract the train set from the whole dataset
dataset['train']

Dataset({
    features: ['id', 'dialogue', 'summary', 'topic'],
    num_rows: 12460
})

In [5]:
# veiw the data of index 1 of train set
dataset['train'][1]

{'id': 'train_1',
 'dialogue': "#Person1#: Hello Mrs. Parker, how have you been?\n#Person2#: Hello Dr. Peters. Just fine thank you. Ricky and I are here for his vaccines.\n#Person1#: Very well. Let's see, according to his vaccination record, Ricky has received his Polio, Tetanus and Hepatitis B shots. He is 14 months old, so he is due for Hepatitis A, Chickenpox and Measles shots.\n#Person2#: What about Rubella and Mumps?\n#Person1#: Well, I can only give him these for now, and after a couple of weeks I can administer the rest.\n#Person2#: OK, great. Doctor, I think I also may need a Tetanus booster. Last time I got it was maybe fifteen years ago!\n#Person1#: We will check our records and I'll have the nurse administer and the booster as well. Now, please hold Ricky's arm tight, this may sting a little.",
 'summary': 'Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.',
 'topic': 'vaccines'}

In [6]:
# check the summary of index 1 data of train set
dataset['train'][1]['summary']

'Mrs Parker takes Ricky for his vaccines. Dr. Peters checks the record and then gives Ricky a vaccine.'

### Inspecting Sample Dialogues and Summaries

In this cell, we select two example indices from the test set (`28` and `240`) to preview the data.

- `example_indices` stores the IDs of the dialogues we want to inspect.
- `dash_line` creates a visual separator for cleaner output formatting.
- The loop prints:
  - A header for each example
  - The **dialogue** from the dataset
  - The corresponding **human-written summary**

This helps us understand the structure and content of the dataset before applying the model for summarization.

In [7]:
example_indices = [28, 240]

dash_line = '-'.join('' for x in range(100))

for i, index in enumerate(example_indices):
    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print('INPUT DIALOGUE:')
    print(dataset['test'][index]['dialogue'])
    print(dash_line)
    print('BASELINE HUMAN SUMMARY:')
    print(dataset['test'][index]['summary'])
    print(dash_line)
    print()

---------------------------------------------------------------------------------------------------
Example  1
---------------------------------------------------------------------------------------------------
INPUT DIALOGUE:
#Person1#: Who stands out in your mind as a man or woman of sound character?
#Person2#: If I think of famous people, I think of Abraham Lincoln.
#Person1#: He's the US president, who walked five miles just to give a lady her change, isn't he?
#Person2#: That's the one. He also was famous for never giving up on his goals.
#Person1#: That's right. He ran for office quite a few times before he was finally elected.
#Person2#: And I also admire him for his courage in fighting for equal rights.
#Person1#: He had great vision, didn't he?
#Person2#: And humility. I would have liked to meet him personally.
---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person2# admires Abraham Lincoln for his pe

### Loading the Pretrained FLAN-T5 Model

Here we define the model we want to use: **`google/flan-t5-base`**, a popular instruction-tuned sequence-to-sequence model.

`AutoModelForSeq2SeqLM.from_pretrained()` loads the full pretrained model, including its weights and architecture, so it can be used for generating summaries from dialogue inputs.

This model will later process tokenized text and produce output summaries.

In [8]:
model_name='google/flan-t5-base'
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

### Loading the Tokenizer

The tokenizer for **FLAN-T5-Base** is loaded using `AutoTokenizer.from_pretrained()`.

- It converts input text into numerical tokens that the model can understand.
- `use_fast=True` enables the optimized “fast” tokenizer implementation for better performance.

Using the same tokenizer that the model was trained with ensures consistent and correct tokenization during inference.

In [9]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True) # use tokenier used in training of this model 'flan-t5-base'

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

### Encoding and Decoding a Sample Sentence

This demonstrates how the tokenizer processes text:

1. **Input sentence**: `"How old are you, David?"`
2. **Encoding**:  
   - `tokenizer(sentence, return_tensors='pt')` converts the text into token IDs in PyTorch tensor format.
   - These token IDs are what the model uses for computation.
3. **Decoding**:  
   - `tokenizer.decode()` converts the token IDs back into human-readable text.
   - `skip_special_tokens=True` removes any model-specific control tokens.

Finally, the encoded token IDs and the decoded sentence are printed to verify that tokenization works correctly.


In [10]:
sentence = "How old are you, David?"

sentence_encoded = tokenizer(sentence, return_tensors='pt') #use pytorch tensor format

sentence_decoded = tokenizer.decode(
        sentence_encoded["input_ids"][0],
        skip_special_tokens=True
    )

print('ENCODED SENTENCE:')
print(sentence_encoded["input_ids"][0])
print('\nDECODED SENTENCE:')
print(sentence_decoded)

ENCODED SENTENCE:
tensor([ 571,  625,   33,   25,    6, 1955,   58,    1])

DECODED SENTENCE:
How old are you, David?


### Baseline Model Generation (No Prompt Engineering)

In this step, we test how the model summarizes dialogues **without giving any instructions**.

For each selected example:

1. **Extract dialogue and its human summary** from the test set.
2. **Tokenize the dialogue** using the FLAN-T5 tokenizer.
3. **Generate a summary** directly from the model using:
   - `model.generate()` with a limit of `max_new_tokens=50`.
4. **Decode the model output** back into text.
5. **Print**:
   - The original dialogue  
   - The human-written summary  
   - The model-generated summary  

This gives us a baseline to compare how well the model performs with no prompt engineering.

In [11]:
for i, index in enumerate(example_indices):
    dialogue = dataset['test'][index]['dialogue']
    summary = dataset['test'][index]['summary']

    inputs = tokenizer(dialogue, return_tensors='pt') # tokenize/ encode
    output = tokenizer.decode( # decode the tokenized output produced from the model
        model.generate(
            inputs["input_ids"], #use the model on tokenized input to generate output produced from the model
            max_new_tokens=50, # set output maximum token at 50.
        )[0],
        skip_special_tokens=True
    )

    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print(f'INPUT PROMPT:\n{dialogue}')
    print(dash_line)
    print(f'BASELINE HUMAN SUMMARY:\n{summary}')
    print(dash_line)
    print(f'MODEL GENERATION - WITHOUT PROMPT ENGINEERING:\n{output}\n')

---------------------------------------------------------------------------------------------------
Example  1
---------------------------------------------------------------------------------------------------
INPUT PROMPT:
#Person1#: Who stands out in your mind as a man or woman of sound character?
#Person2#: If I think of famous people, I think of Abraham Lincoln.
#Person1#: He's the US president, who walked five miles just to give a lady her change, isn't he?
#Person2#: That's the one. He also was famous for never giving up on his goals.
#Person1#: That's right. He ran for office quite a few times before he was finally elected.
#Person2#: And I also admire him for his courage in fighting for equal rights.
#Person1#: He had great vision, didn't he?
#Person2#: And humility. I would have liked to meet him personally.
---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person2# admires Abraham Lincoln for his pers

The  results show that **the model performs poorly when no instructions are provided**. FLAN-T5 receives only the raw dialogue and is expected to infer that it should summarize the conversation, but:

- The generated output does **not resemble a summary**.
- In the first example, the model responds as if continuing the conversation rather than summarizing it.
- In the second example, the output becomes a **partial, incorrect continuation** instead of a concise summary.
- The generated text does not capture the key information found in the human-written summaries.

This demonstrates that **instruction prompts are necessary** for guiding the model toward producing meaningful and accurate summaries.

### Zero-Shot Summarization with Instruction Prompt

Now, we apply **zero-shot prompting** to improve model summaries:

1. **Construct an instruction prompt**:
This explicitly tells the model to generate a summary rather than continue the dialogue.
2. **Tokenize the prompt** and pass it to the model.
3. **Generate output** with `max_new_tokens=50` and decode it back to text.
4. **Print**:
- The full prompt
- The human-written summary
- The model-generated summary

By providing a clear instruction, the model is guided to produce a concise summary of the dialogue, improving over the baseline with no prompt engineering.

In [12]:
# Summarize Dialogue with an instruction prompt , Zero shot inference
for i, index in enumerate(example_indices):
    dialogue = dataset['test'][index]['dialogue']
    summary = dataset['test'][index]['summary']

    prompt = f"""
Summarize the following conversation.

{dialogue}

Summary:
    """ # instruction prompt for the model

    # Input constructed prompt instead of the raw dialogue.
    inputs = tokenizer(prompt, return_tensors='pt')
    output = tokenizer.decode(
        model.generate(
            inputs["input_ids"],
            max_new_tokens=50,
        )[0],
        skip_special_tokens=True
    )

    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print(f'INPUT PROMPT:\n{prompt}')
    print(dash_line)
    print(f'BASELINE HUMAN SUMMARY:\n{summary}')
    print(dash_line)
    print(f'MODEL GENERATION - ZERO SHOT:\n{output}\n')

---------------------------------------------------------------------------------------------------
Example  1
---------------------------------------------------------------------------------------------------
INPUT PROMPT:
                       
Summarize the following conversation.

#Person1#: Who stands out in your mind as a man or woman of sound character?
#Person2#: If I think of famous people, I think of Abraham Lincoln.
#Person1#: He's the US president, who walked five miles just to give a lady her change, isn't he?
#Person2#: That's the one. He also was famous for never giving up on his goals.
#Person1#: That's right. He ran for office quite a few times before he was finally elected.
#Person2#: And I also admire him for his courage in fighting for equal rights.
#Person1#: He had great vision, didn't he?
#Person2#: And humility. I would have liked to meet him personally.

Summary:                
    
----------------------------------------------------------------------------

The zero-shot instruction prompt **slightly improves the model's output**, but the results are still **not very accurate**:

- The summaries are more like general statements or guesses rather than concise, factual summaries.
- Key details from the dialogues are often **missing or incorrect** (e.g., "near the border" is not mentioned in the original dialogue).
- This shows that while instruction prompts help guide the model, **zero-shot summarization alone may not be sufficient** for highly accurate outputs.  

It highlights the need for **one-shot or few-shot prompting** to provide examples for better context.

### Zero-Shot Summarization with Alternative Prompt

Here, we test a **different zero-shot prompt** to see if rephrasing improves summarization:

- The prompt now frames the task as a question:
- The model is expected to summarize the dialogue by answering the question rather than following a "Summary:" instruction.
- Steps:
1. Tokenize the new prompt.
2. Generate and decode the model output.
3. Print the input prompt, human summary, and model-generated summary.

This approach explores how **prompt wording affects zero-shot performance** and helps compare different prompting strategies.



In [13]:
#Zero shot inference with different prompt
for i, index in enumerate(example_indices):
    dialogue = dataset['test'][index]['dialogue']
    summary = dataset['test'][index]['summary']

    prompt = f"""
Dialogue:

{dialogue}

What was going on?
""" #instructed prompt (different from above one)

    inputs = tokenizer(prompt, return_tensors='pt')
    output = tokenizer.decode(
        model.generate(
            inputs["input_ids"],
            max_new_tokens=50,
        )[0],
        skip_special_tokens=True
    )

    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print(f'INPUT PROMPT:\n{prompt}')
    print(dash_line)
    print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
    print(dash_line)
    print(f'MODEL GENERATION - ZERO SHOT:\n{output}\n')

---------------------------------------------------------------------------------------------------
Example  1
---------------------------------------------------------------------------------------------------
INPUT PROMPT:

Dialogue:

#Person1#: Who stands out in your mind as a man or woman of sound character?
#Person2#: If I think of famous people, I think of Abraham Lincoln.
#Person1#: He's the US president, who walked five miles just to give a lady her change, isn't he?
#Person2#: That's the one. He also was famous for never giving up on his goals.
#Person1#: That's right. He ran for office quite a few times before he was finally elected.
#Person2#: And I also admire him for his courage in fighting for equal rights.
#Person1#: He had great vision, didn't he?
#Person2#: And humility. I would have liked to meet him personally.

What was going on?

---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person2# adm

Using the rephrased prompt improves the model output slightly:

- The generated summaries are **more focused on the dialogue content** compared to the previous zero-shot attempt.
- However, they still **miss some key details** or include minor inaccuracies (e.g., "near the border" is not mentioned in the dialogue).
- This shows that **prompt wording can influence the model's performance**, but zero-shot prompting alone may not reliably produce fully accurate summaries.

### One-Shot Summarization

This cell defines a function `make_prompt()` to create a **one-shot prompt** for the model:

1. **Inputs**:
   - `example_indices_full`: indices of dialogues with their human summaries to use as examples.
   - `example_index_to_summarize`: index of the dialogue we want the model to summarize.
2. **Function Logic**:
   - For each example dialogue, append:
     ```
     Dialogue:
     {dialogue}
     What was going on?
     {summary}
     ```
     This provides the model with a clear input-output example.
   - Then append the **target dialogue** (without its summary) to guide the model to generate a summary.
3. **Purpose**: By including **one labeled example**, the model can better understand the task and produce more accurate summaries, compared to zero-shot prompting.


In [14]:
#one shot inference
def make_prompt(example_indices_full, example_index_to_summarize):
    prompt = ''
    for index in example_indices_full:
        dialogue = dataset['test'][index]['dialogue']
        summary = dataset['test'][index]['summary']

        # The stop sequence '{summary}\n\n\n' is important for FLAN-T5. Other models may have their own preferred stop sequence.
        prompt += f"""
Dialogue:

{dialogue}

What was going on?
{summary}


""" # this prompt include both diagloue and human summary as

    dialogue = dataset['test'][example_index_to_summarize]['dialogue']

    prompt += f"""
Dialogue:

{dialogue}

What was going on?
""" # this include only dialoguto test

    return prompt

Then, We specify **one example** for one-shot learning:
   - `example_indices_full = [28]` → dialogue and summary used as a reference.
   - `example_index_to_summarize = 240` → the dialogue we want the model to summarize.
Call `make_prompt()` to generate the full **one-shot prompt**, which includes:
   - The example dialogue and its human summary.
   - The target dialogue for summarization.
Print the prompt to inspect its structure before feeding it to the model.

This shows how one-shot prompting combines **demonstration and test input** to guide the model.


In [15]:
example_indices_full = [28]
example_index_to_summarize = 240

one_shot_prompt = make_prompt(example_indices_full, example_index_to_summarize)

print(one_shot_prompt)


Dialogue:

#Person1#: Who stands out in your mind as a man or woman of sound character?
#Person2#: If I think of famous people, I think of Abraham Lincoln.
#Person1#: He's the US president, who walked five miles just to give a lady her change, isn't he?
#Person2#: That's the one. He also was famous for never giving up on his goals.
#Person1#: That's right. He ran for office quite a few times before he was finally elected.
#Person2#: And I also admire him for his courage in fighting for equal rights.
#Person1#: He had great vision, didn't he?
#Person2#: And humility. I would have liked to meet him personally.

What was going on?
#Person2# admires Abraham Lincoln for his perseverance, courage and humility.



Dialogue:

#Person1#: Hello. Is this ABC Rent-a-car Company?
#Person2#: Yes, speaking. May I help you?
#Person1#: This morning we rented a car and we are on the way to Niagara Falls. I'm afraid we have a car accident near the border.
#Person2#: That's too bad. What kind of accident

### One-Shot Summarization Inference

In this cell, we use the **one-shot prompt** to generate a summary:

1. Retrieve the **human summary** for the target dialogue (`example_index_to_summarize`) for comparison.
2. **Tokenize the one-shot prompt** (which includes one example dialogue-summary pair and the target dialogue).
3. **Generate output** from the model with `max_new_tokens=50`.
4. **Decode** the token IDs into human-readable text.
5. **Print**:
   - The baseline human summary
   - The model-generated summary

By providing a single example, the model is expected to produce a **more accurate and contextually relevant summary** than in zero-shot scenarios.

In [16]:
summary = dataset['test'][example_index_to_summarize]['summary'] # the input contain dialogue and human summary of index 28 as example and diaglogue of index 240,
                                                                  #the model will generate the summary of the index 240 dialogue

inputs = tokenizer(one_shot_prompt, return_tensors='pt')
output = tokenizer.decode(
    model.generate(
        inputs["input_ids"],
        max_new_tokens=50,
    )[0],
    skip_special_tokens=True
)

print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - ONE SHOT:\n{output}')

---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# rent a car from ABC Rent-a-car Company this morning and met an accident. #Person2# will call an ambulance and police for #Person1#.

---------------------------------------------------------------------------------------------------
MODEL GENERATION - ONE SHOT:
Person1 is in a car accident near the border.


The one-shot prompt improves the model's understanding compared to zero-shot:

- The model output **captures the main event** (car accident) from the dialogue.
- However, it **misses some key details**, such as calling an ambulance and police, which are present in the human summary.
- This shows that **even a single example helps guide the model**, but multiple examples (few-shot) may further improve accuracy and completeness.


### Few-Shot Summarization

Now, we extend the one-shot approach to **few-shot prompting**:

1. **Select multiple examples**:
   - `example_indices_full = [40, 28, 125]` → each example includes a dialogue and its human-written summary.
2. **Target dialogue**:
   - `example_index_to_summarize = 240` → the dialogue we want the model to summarize.
3. Call `make_prompt()` to generate a **few-shot prompt**, which concatenates multiple examples followed by the target dialogue.
4. Print the prompt to inspect its structure.

Few-shot prompting provides the model with **more context and examples**, which typically improves summarization quality over one-shot.


In [17]:
#few shot inference
example_indices_full = [40, 28, 125]
example_index_to_summarize = 240

few_shot_prompt = make_prompt(example_indices_full, example_index_to_summarize) # both dialogue and human summary of index 40,28,120 will be contained as example
                                                                                # diaglogue of index 240 is used for model to generate summary

print(few_shot_prompt)


Dialogue:

#Person1#: What time is it, Tom?
#Person2#: Just a minute. It's ten to nine by my watch.
#Person1#: Is it? I had no idea it was so late. I must be off now.
#Person2#: What's the hurry?
#Person1#: I must catch the nine-thirty train.
#Person2#: You've plenty of time yet. The railway station is very close. It won't take more than twenty minutes to get there.

What was going on?
#Person1# is in a hurry to catch a train. Tom tells #Person1# there is plenty of time.



Dialogue:

#Person1#: Who stands out in your mind as a man or woman of sound character?
#Person2#: If I think of famous people, I think of Abraham Lincoln.
#Person1#: He's the US president, who walked five miles just to give a lady her change, isn't he?
#Person2#: That's the one. He also was famous for never giving up on his goals.
#Person1#: That's right. He ran for office quite a few times before he was finally elected.
#Person2#: And I also admire him for his courage in fighting for equal rights.
#Person1#: He h

### Few-Shot Summarization Inference

This cell uses the **few-shot prompt** to generate a summary:

1. Retrieve the **human summary** for the target dialogue for comparison.
2. **Tokenize the few-shot prompt** (including multiple example dialogue-summary pairs and the target dialogue).
3. **Generate output** from the model with `max_new_tokens=50`.
4. **Decode** the model output into readable text.
5. **Print**:
   - The baseline human summary
   - The model-generated summary

By providing several examples, the model is expected to produce a **more accurate and contextually complete summary** than one-shot or zero-shot prompting.

In [18]:
summary = dataset['test'][example_index_to_summarize]['summary']

inputs = tokenizer(few_shot_prompt, return_tensors='pt') # few shot prompt is input to the model
output = tokenizer.decode(
    model.generate(
        inputs["input_ids"],
        max_new_tokens=50,
    )[0],
    skip_special_tokens=True
)

print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - FEW SHOT:\n{output}')

Token indices sequence length is longer than the specified maximum sequence length for this model (915 > 512). Running this sequence through the model will result in indexing errors


---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# rent a car from ABC Rent-a-car Company this morning and met an accident. #Person2# will call an ambulance and police for #Person1#.

---------------------------------------------------------------------------------------------------
MODEL GENERATION - FEW SHOT:
Person1 is in a car accident near the border. He is calling the police and the ambulance.


The few-shot prompt significantly improves the model's output compared to zero-shot and one-shot:

- The generated summary **captures both the main event and additional details**, such as calling the ambulance and police.
- It is **closer to the human-written summary** in terms of completeness and relevance.
- This demonstrates that providing **multiple examples helps the model better understand the task** and produce more accurate and context-aware summaries.


### Conclusion

This project demonstrates the use of **FLAN-T5** for dialogue summarization on the **DialogSum dataset**. Key observations from the experiments:

- **Baseline (no prompt)**: The model struggles and produces irrelevant or incomplete summaries.
- **Zero-shot prompting**: Slight improvements occur when giving instructions, but results remain inconsistent.
- **One-shot prompting**: Including a single example helps the model capture the main events but may miss some details.
- **Few-shot prompting**: Providing multiple examples produces the most accurate and contextually complete summaries, closely matching human-written summaries.

Overall, the project highlights the **importance of prompt design and example-based guidance** in improving LLM performance for summarization tasks.